# DL · Encuentro 3 · Secuencias, atención y Transformer

Secuencias, atención y contexto

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [El orden puede cambiar el significado](#scrollTo=tema-dl-secuencias)
2. [Relacionar elementos con contexto](#scrollTo=tema-dl-atencion)
3. [Atención más información de orden](#scrollTo=tema-dl-transformers)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Pronosticar una lectura con LSTM

**Lo que vas a poder hacer:** Explicar por qué el orden importa y distinguir un estado calculado de un pronóstico.

### Entiende la idea

Una secuencia es una lista en la que el orden tiene significado. Una red recurrente actualiza un estado, un resumen del pasado, con cada entrada. Una LSTM (Long Short-Term Memory, memoria de largo y corto plazo) aprende mecanismos que regulan qué conservar e incorporar. Primero usaremos una regla fija para observar el papel del orden.

**Secuencia:** Entradas organizadas por tiempo u otro orden definido; intercambiarlas puede cambiar la interpretación.

**Estado:** Resumen que se actualiza al leer una entrada. No tiene por qué ser la predicción final.

**RNN y LSTM:** RNN: Recurrent Neural Network, red neuronal recurrente. LSTM es un tipo de RNN que aprende a regular la información conservada.

### Un caso pequeño antes del programa

**¿Por qué los mismos valores pueden dejar estados finales diferentes?**

Una bodega recibe 10, 20 y 30 cajas en tres horas. Comparamos ese recorrido con recibir 30, 20 y 10. Ambos tienen media 20.

**Cómo leerlo:** En cada paso se mezcla mitad del estado anterior y mitad de la entrada actual. Sigue las flechas en orden: la última entrada influye de manera distinta según cuándo llega.

**Para recordar:** La media pierde información de orden. Este estado conserva parte de ella con una regla dada; todavía no pronostica la siguiente entrega.

### Ahora identifica los datos de esta práctica

Leer un pronóstico de un paso hecho con doce observaciones anteriores.

1. En dl-encuentro-3.ipynb, Tema 1, ejecuta su preparación y la LSTM.
2. Identifica qué doce lecturas anteriores forman la entrada y qué lectura siguiente es el objetivo.
3. Compara el error de LSTM y la referencia de último valor sobre los mismos instantes reservados.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** La regla de mitades del aula explica memoria con números fáciles. La LSTM sí aprende sus parámetros y usa otra serie; sus cifras no deben coincidir.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Pronosticar la siguiente lectura con LSTM

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Una LSTM real para una secuencia pequeña y artificial.
r=np.random.default_rng(SEMILLA);tiempo=np.arange(240);serie=np.sin(tiempo*.12)+.12*r.normal(size=len(tiempo))
VENTANA=12;CORTE=180
entradas=np.array([serie[i-VENTANA:i] for i in range(VENTANA,len(serie))],dtype='float32')
objetivos=serie[VENTANA:].astype('float32');instante=tiempo[VENTANA:]
train=instante<CORTE;test=~train
xt=torch.tensor(entradas[train,:,None]);yt=torch.tensor(objetivos[train,None]);xe=torch.tensor(entradas[test,:,None])
class PredictorLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.memoria=nn.LSTM(1,12,batch_first=True);self.salida=nn.Linear(12,1)
    def forward(self,x):
        secuencia,_=self.memoria(x);return self.salida(secuencia[:,-1])
torch.manual_seed(SEMILLA);lstm=PredictorLSTM();opt=torch.optim.Adam(lstm.parameters(),lr=.02)
for epoca in range(90):
    opt.zero_grad();error=nn.functional.mse_loss(lstm(xt),yt);error.backward();opt.step()
lstm.eval()
with torch.no_grad():p=lstm(xe).numpy().ravel()
baseline=entradas[test,-1]
print('MAE LSTM:',round(mean_absolute_error(objetivos[test],p),3),'· MAE último valor:',round(mean_absolute_error(objetivos[test],baseline),3))
plt.plot(instante[test],objetivos[test],label='Real simulado',color=GRIS);plt.plot(instante[test],p,label='LSTM',color=AZUL);plt.plot(instante[test],baseline,label='Último valor',color=VERDE,alpha=.6)
plt.legend();plt.xlabel('Tiempo');plt.ylabel('Lectura simulada (sin unidades)');plt.title('Pronóstico de un paso usando la ventana disponible');mostrar('11-lstm')
print('Se evalúa un paso hacia adelante; no es un pronóstico recursivo de toda la serie.')


### Después de ejecutar: compara en los mismos instantes

La línea real es la respuesta reservada; la LSTM y Último valor son dos propuestas para el mismo instante. Busca una zona de subida y otra de bajada y compara cuánto se separan. El MAE resume esas distancias. Repetir el último valor es una referencia que el modelo puede o no mejorar.


### Lee, comprueba y explica

**Qué debes poder comprobar:** La práctica usa otra serie artificial. No trasladas los estados de cajas al pronóstico ni interpretas sus sesenta casos de prueba como un único pronóstico a sesenta pasos.

**Prueba pequeña:** La regla de mitades del aula explica memoria con números fáciles. La LSTM sí aprende sus parámetros y usa otra serie; sus cifras no deben coincidir.

**Pregunta:** ¿El estado final 21,25 es una predicción de las cajas que llegarán después?

**Respuesta razonada:** No. Es el resumen de las tres entradas aplicando la regla elegida. Para pronosticar hace falta definir un objetivo futuro y comprobar las predicciones con datos reservados.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-secuencias)

[Siguiente tema: Relacionar elementos con contexto](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-atencion)


## Tema 2 · Mezclar dos registros con atención

**Lo que vas a poder hacer:** Seguir cómo una consulta determina participaciones y cómo estas producen una mezcla.

### Entiende la idea

La atención combina información dando distinta participación a cada fuente. Compara una consulta con las claves de las fuentes, transforma las compatibilidades en pesos que suman uno y mezcla sus valores. Una participación mayor acerca la mezcla a esa fuente. En una red, las representaciones que permiten estas comparaciones pueden aprenderse.

**Consulta Q y claves K:** La consulta expresa qué se busca; las claves permiten comparar las fuentes. Aquí son listas de números dadas.

**Valores V:** Información que se mezcla después de calcular cuánto participa cada fuente.

**Pesos de atención:** Participaciones recalculadas para una consulta. No son los parámetros aprendidos ni prueban qué causa un resultado.

### Un caso pequeño antes del programa

**¿Cómo se obtiene una mezcla diferente sin cambiar las dos fuentes?**

Los valores son A=10 y B=20 cajas ficticias. Consulta y claves fueron construidas para observar el reparto; no provienen de pedidos reales.

**Cómo leerlo:** La consulta inicial da a A aproximadamente dos tercios de la participación y a B un tercio. Multiplica cada valor por su participación y suma: la mezcla queda cerca de 13,3 cajas.

**Para recordar:** Comparar fuentes y mezclar su contenido son dos operaciones diferentes. Aquí se calcula atención; no se entrena un Transformer.

### Ahora identifica los datos de esta práctica

Reconstruir consulta, pesos y mezcla con las dos fuentes del ejemplo.

1. En dl-encuentro-3.ipynb, Tema 2, ejecuta su preparación y la atención de dos registros.
2. Lee por separado la comparación con las claves, el reparto calculado y la combinación de valores.
3. Contrasta el reparto mostrado con la mezcla; usa los valores sin redondear para comprobar la cuenta.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Con participaciones iguales, la mezcla debe ser 15. Un peso mayor para A la acerca a 10; un peso mayor para B, a 20.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Consulta, claves y valores son símbolos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **Q, K y V** — Q, K y V: Consulta, clave y valor. Q identifica la consulta; K, las claves con las que se compara; V, la información que se mezcla. Son símbolos de cantidades, no tres siglas de algoritmos. La comparación Q–K determina pesos; la combinación de V produce la salida, sin demostrar causalidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)

- **Softmax** — Softmax: Transformación de puntuaciones en participaciones. Para cada puntuación calcula su exponencial y la divide entre la suma de exponenciales del grupo. Así obtiene valores positivos que suman uno cuando las entradas son finitas. La normalización se hace sobre un eje elegido; en atención, por las fuentes de cada consulta. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

- **Transformer** — Transformer: Arquitectura de red basada en atención. Arquitectura que combina atención con otras transformaciones, conexiones residuales y normalización. Incorpora información de posición para distinguir órdenes en la tarea del cuaderno. No es una sigla, una sola operación ni necesariamente un chatbot. [Fuente](https://arxiv.org/abs/1706.03762)


### Lee una mezcla antes de leer sus símbolos

Si tomas la mitad de 10 y la mitad de 20, tienes 5+10=15. En la cuenta de atención, una consulta [1,0] da aproximadamente 67 % de participación a A y 33 % a B: la mezcla queda cerca de 13,3, más próxima a 10.

**Vector** significa lista ordenada de números. [1,0] tiene dos posiciones; no es una fracción. **Producto punto** multiplica números del mismo lugar y suma: [1,0] con [1,0] da 1×1 + 0×0 = 1. Se divide por la raíz de 2 porque las listas tienen dos posiciones. **Softmax** convierte esas comparaciones en participaciones positivas que suman 1.

**Tip:** la consulta [0,0] compara igual con A y B. Comprueba que su reparto sea 0,5 y 0,5 y que la salida sea 15. Atención calcula una representación; 13,3 no es una orden de compra ni una cantidad aprendida de ventas.

En el Transformer siguiente sí se ajustan parámetros usando ejemplos de una tarea distinta. Compara sus respuestas con las cuatro etiquetas que resolviste. **Etiqueta** es la respuesta correcta conocida; **predicción** es lo que calcula la red. Una puntuación alta de la red todavía puede equivocarse.


### Mezclar dos registros con atención

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Dos registros como en el aula: mezclamos cantidades con pesos dados por la comparación.
fuentes=['A: 10','B: 20']
consultas=['Q=[1,0]','Q=[0,1]','Q=[0,0]']
Q=np.array([[1.,0.],[0.,1.],[0.,0.]])
K=np.array([[1.,0.],[0.,1.]])
V=np.array([[10.],[20.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True))
atencion=exp/exp.sum(axis=1,keepdims=True)
mezclas=atencion@V
assert np.allclose(atencion.sum(axis=1),1)
assert np.allclose(atencion[2],[.5,.5]) and np.isclose(mezclas[2,0],15)
for consulta,pesos,mezcla in zip(consultas,atencion,mezclas[:,0]):
    print(consulta,'· participación A/B:',np.round(pesos,4),'· mezcla:',round(float(mezcla),3))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(atencion,cmap='Blues',vmin=0,vmax=1)
axs[0].set_xticks(range(2),fuentes);axs[0].set_yticks(range(3),consultas)
for i in range(3):
    for j in range(2):axs[0].text(j,i,f'{atencion[i,j]*100:.1f} %',ha='center',va='center',color='black')
axs[0].set(title='Cuánto participa cada fuente',xlabel='Fuente',ylabel='Consulta dada')
axs[1].bar(consultas,mezclas[:,0],color=[AZUL,VERDE,GRIS]);axs[1].set(title='Resultado de cada mezcla',ylabel='Valor mezclado',ylim=(0,22))
axs[1].axhline(10,ls=':',color='gray');axs[1].axhline(20,ls=':',color='gray')
mostrar('12-atencion')


### Después de ejecutar: comprueba el reparto

Cada fila debe sumar 100 %. Q=[0,0] produce partes iguales y mezcla 15. Q=[1,0] da más participación a A=10 y una salida cercana a 13,3. No se entrenó atención aquí: Q, K y V fueron elegidos para enseñar la operación.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Los pesos suman uno. Puedes distinguir la consulta y las claves sin unidades físicas de los valores y la mezcla expresados en cajas ficticias.

**Prueba pequeña:** Con participaciones iguales, la mezcla debe ser 15. Un peso mayor para A la acerca a 10; un peso mayor para B, a 20.

**Pregunta:** Si ambas fuentes participan por mitades, ¿la salida es cero o un valor intermedio?

**Respuesta razonada:** Es 15: la mitad de 10 más la mitad de 20. Una consulta que produzca el mismo peso para ambas fuentes no elimina sus valores.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-atencion)

[Siguiente tema: Atención más información de orden](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transformers)


## Tema 3 · Comprobar el orden con un Transformer

**Lo que vas a poder hacer:** Ubicar representación, posición, atención y salida dentro de un Transformer.

### Entiende la idea

Un Transformer organiza bloques de atención y pequeñas redes por posición para construir representaciones con contexto. Las entradas se convierten en vectores; la atención relaciona elementos y las transformaciones procesan sus representaciones. La información de orden y la forma de leer las posiciones permiten usar su ubicación. La salida depende de la tarea elegida.

**Token y representación:** Un token es una unidad de entrada. En esta práctica es un número, que se convierte en una lista de valores llamada vector.

**Autoatención:** Relaciona elementos de la misma secuencia y mezcla información entre ellos.

**Posición y salida:** La posición informa dónde está cada elemento; la salida transforma el conjunto en la respuesta de la tarea.

### Un caso pequeño antes del programa

**¿Cómo se organiza una red que debe responder sobre una secuencia?**

La tarea construida pregunta si la última cantidad de cuatro horas es menor que la primera. Primero calculamos nosotros la etiqueta; después comprobamos qué responde la red.

**Cómo leerlo:** Sigue las piezas de la arquitectura. En 1→2→3→4 la etiqueta es 0; en 4→3→2→1 es 1. Estas etiquetas conocidas no son todavía predicciones del modelo.

**Para recordar:** Atención es una pieza del Transformer. Conocer una regla sencilla permite comprobar su respuesta, pero no demuestra que la red ya la aprendió.

### Ahora identifica los datos de esta práctica

Relacionar las piezas de un Transformer entrenado con una tarea y una prueba comprobables.

1. En dl-encuentro-3.ipynb, Tema 3, ejecuta su preparación y el Transformer de la tarea de orden.
2. Localiza representación, posición, atención y salida; después lee la evaluación reservada que calcula.
3. Compara las cuatro respuestas esperadas con las respuestas predichas que imprime, sin dar por hecho que coinciden.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** En 3→1→7→2 la respuesta conocida es 1: 2 es menor que 3. La respuesta calculada por la red puede equivocarse; se comprueba al ejecutar, sin prometer aciertos.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Comprobar el orden con un Transformer

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Transformer real, pequeño, entrenado para una tarea de orden.
r=np.random.default_rng(SEMILLA);seq=r.integers(1,8,size=(700,4));seq=seq[seq[:,0]!=seq[:,-1]]
seq=np.unique(seq,axis=0) # reserva de patrones distintos, no solo de filas
y=(seq[:,0]>seq[:,-1]).astype('int64')
Xtr,Xte,ytr,yte=train_test_split(seq,y,test_size=.25,stratify=y,random_state=SEMILLA)
compartidas=set(map(tuple,Xtr)) & set(map(tuple,Xte))
assert not compartidas, 'Entrenamiento y prueba comparten una secuencia.'
print('Secuencias distintas:',len(seq),'· entrenamiento:',len(Xtr),'· prueba:',len(Xte))
print('Clases [0, 1] en entrenamiento:',np.bincount(ytr),'· en prueba:',np.bincount(yte))
class OrdenTransformer(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(8,16);self.pos=nn.Parameter(torch.randn(1,4,16)*.1)
        capa=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(capa,num_layers=1,enable_nested_tensor=False);self.salida=nn.Linear(4*16,2)
    def forward(self,x):
        # La salida por posiciones también conserva orden; no aislamos el efecto de self.pos.
        h=self.encoder(self.emb(x)+self.pos);return self.salida(h.flatten(1))
torch.manual_seed(SEMILLA);transformer=OrdenTransformer();opt=torch.optim.Adam(transformer.parameters(),lr=.005)
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=64,shuffle=True)
hist=[]
for epoca in range(45):
    transformer.train();suma=0
    for bx,by in loader:
        opt.zero_grad();error=nn.functional.cross_entropy(transformer(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(Xtr))
transformer.eval()
with torch.no_grad():
    pred=transformer(torch.tensor(Xte)).argmax(1).numpy()
    ejemplos=torch.tensor([[1,2,3,4],[4,3,2,1],[2,7,1,3],[3,1,7,2]]);prob=torch.softmax(transformer(ejemplos),dim=1).numpy()
print('Accuracy en prueba:',round(accuracy_score(yte,pred),3))
esperadas=(ejemplos[:,0]>ejemplos[:,-1]).numpy().astype(int)
calculadas=prob.argmax(axis=1)
for secuencia,esperada,calculada,p1 in zip(ejemplos.tolist(),esperadas,calculadas,prob[:,1]):
    print(secuencia,'· respuesta conocida:',int(esperada),'· respuesta de la red:',int(calculada),'· puntuación clase 1:',round(float(p1),3))
print('Comparar extremos no pregunta si toda la secuencia baja. Las respuestas de la red pueden fallar.')
plt.plot(hist,color=AZUL);plt.xlabel('Época');plt.ylabel('Pérdida');plt.title('Pérdida de entrenamiento en la tarea de orden');mostrar('12-transformer')


### Después de ejecutar: respuesta correcta y calculada

La etiqueta de los cuatro ejemplos se puede comprobar mirando solo primero y último. La red se comprueba contra esa etiqueta; si falla, el resultado sigue siendo un error aunque su puntuación sea alta. El porcentaje de prueba resume otros patrones reservados y no certifica tareas de lenguaje.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Distingues la etiqueta de referencia de las dos puntuaciones calculadas y de la clase elegida. La prueba del cuaderno excluye secuencias con extremos iguales.

**Prueba pequeña:** En 3→1→7→2 la respuesta conocida es 1: 2 es menor que 3. La respuesta calculada por la red puede equivocarse; se comprueba al ejecutar, sin prometer aciertos.

**Pregunta:** ¿Qué aporta la atención y qué aporta la posición en este recorrido?

**Respuesta razonada:** La atención mezcla información entre elementos según sus representaciones; la posición informa dónde están. El modelo completo transforma esas representaciones en puntuaciones de clase. En el cuaderno, la lectura final también conserva posiciones: no se aísla el efecto de añadir información posicional.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transformers)

[Siguiente tema: De una foto a un registro](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-yolo)


## Cierra el encuentro con una explicación

1. **El orden puede cambiar el significado:** La media pierde información de orden. Este estado conserva parte de ella con una regla dada; todavía no pronostica la siguiente entrega.
2. **Relacionar elementos con contexto:** Comparar fuentes y mezclar su contenido son dos operaciones diferentes. Aquí se calcula atención; no se entrena un Transformer.
3. **Atención más información de orden:** Atención es una pieza del Transformer. Conocer una regla sencilla permite comprobar su respuesta, pero no demuestra que la red ya la aprendió.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
